In [1]:
import pandas as pd

read and explore mtsamples file

In [2]:
df_notes= pd.read_csv("data/mtsamples.csv")
print(df_notes.shape)
print(df_notes.head(2))

(4999, 6)
   Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   

       medical_specialty                                sample_name  \
0   Allergy / Immunology                         Allergic Rhinitis    
1             Bariatrics   Laparoscopic Gastric Bypass Consult - 2    

                                       transcription  \
0  SUBJECTIVE:,  This 23-year-old white female pr...   
1  PAST MEDICAL HISTORY:, He has difficulty climb...   

                                            keywords  
0  allergy / immunology, allergic rhinitis, aller...  
1  bariatrics, laparoscopic gastric bypass, weigh...  


In [3]:
df_notes["combined_text"]= df_notes["description"].fillna("")+ df_notes["transcription"].fillna("")
print(df_notes.shape)

(4999, 7)


read and explore icd code file

In [4]:
code_list=[]
with open ("data/icd10cm-codes-2026.txt", "r", encoding="utf-8") as f:
    for line in f:
        line=line.strip()
        if line:
            code,description=line.split(maxsplit=1)
            code_list.append((code,description))
code_df=pd.DataFrame(code_list, columns=["code","code_description"])
print(code_df.shape)


(74719, 2)


# Using sapBERT model for tokenization

In [5]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)

device= torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
model.eval()
print(device)

/opt/miniconda3/envs/dl_projects/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mps


converting code text to vectors:
-part 1 converting the code text into list and then into batches
-part 2 these batches are given to tokenizer to get token id and then to model to get vectors

In [6]:
code_description= code_df["code_description"].tolist()
batch_size=32
print(len(code_description))
print(code_description[0])

74719
Cholera due to Vibrio cholerae 01, biovar cholerae


testing for a single batch

In [7]:

for i in range(0,len(code_description),batch_size):
    batch=code_description[i:i+batch_size]
    print(f"Each batch shape:{i,len(batch)}")
    
# converting to token of a single batch  
sample_batch=code_description[0:batch_size]
token_id = tokenizer(sample_batch,padding=True, truncation=True,return_tensors="pt")
print(f'input ids of token_id of sample batch : {token_id["input_ids"].shape}')

# token id converted into vector
token_id= token_id.to(device)
with torch.no_grad():
    output = model(**token_id)
cls_vectors = output.last_hidden_state[:, 0, :]
print(f"cls_vectors of sample batch:{cls_vectors.shape}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Each batch shape:(0, 32)
Each batch shape:(32, 32)
Each batch shape:(64, 32)
Each batch shape:(96, 32)
Each batch shape:(128, 32)
Each batch shape:(160, 32)
Each batch shape:(192, 32)
Each batch shape:(224, 32)
Each batch shape:(256, 32)
Each batch shape:(288, 32)
Each batch shape:(320, 32)
Each batch shape:(352, 32)
Each batch shape:(384, 32)
Each batch shape:(416, 32)
Each batch shape:(448, 32)
Each batch shape:(480, 32)
Each batch shape:(512, 32)
Each batch shape:(544, 32)
Each batch shape:(576, 32)
Each batch shape:(608, 32)
Each batch shape:(640, 32)
Each batch shape:(672, 32)
Each batch shape:(704, 32)
Each batch shape:(736, 32)
Each batch shape:(768, 32)
Each batch shape:(800, 32)
Each batch shape:(832, 32)
Each batch shape:(864, 32)
Each batch shape:(896, 32)
Each batch shape:(928, 32)
Each batch shape:(960, 32)
Each batch shape:(992, 32)
Each batch shape:(1024, 32)
Each batch shape:(1056, 32)
Each batch shape:(1088, 32)
Each batch shape:(1120, 32)
Each batch shape:(1152, 32)
E

tokenization and converting to vectors for all samples

In [8]:
all_code_vectors = []

with torch.no_grad():
    for i in range(0, len(code_description), batch_size):
        batch = code_description[i : i + batch_size]
        #  tokenize
        token_id = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        # vectorize
        output = model(**token_id)
        batch_vectors = output.last_hidden_state[:, 0, :]
        all_code_vectors.append(batch_vectors.cpu())   

code_vectors = torch.cat(all_code_vectors, dim=0)
code_vectors = code_vectors.numpy()   
print(code_vectors.shape)

(74719, 768)


tokenization and vectorization for notes description

In [9]:
combined_text= df_notes["combined_text"].tolist()
print(len(combined_text))

4999


In [10]:
all_notes_vectors=[]
with torch.no_grad():
    for i in range(0,len(combined_text),batch_size):
        batch= combined_text[i:i+batch_size]
        # tokenize
        token_id = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        # vectorize
        output= model(**token_id)
        batch_vectors = output.last_hidden_state[ :, 0, :]
        all_notes_vectors.append(batch_vectors.cpu())
        
notes_vectors= torch.cat(all_notes_vectors, dim=0)
notes_vectors= notes_vectors.numpy()
print(notes_vectors.shape)

(4999, 768)


comparing notes and code vectors

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix =cosine_similarity(notes_vectors, code_vectors)
# similairty matrix highest index vaue and highest score similaity in a row
best_match_index = similarity_matrix.argmax(axis=1)
best_match_score = similarity_matrix.max(axis=1)

df_notes["matched_code"]= code_df["code"].values[best_match_index]
df_notes["matching_score"] = best_match_score

print(df_notes.shape)
print(df_notes.head(5))

(4999, 9)
   Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   
2           2           Consult for laparoscopic gastric bypass.   
3           3                             2-D M-Mode. Doppler.     
4           4                                 2-D Echocardiogram   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3   Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4   Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                       transcription  \
0  SUBJECTIVE:,  This 23-year-old white female pr...   
1  PAS

In [12]:
print(df_notes["matching_score"].describe())

count    4999.000000
mean        0.598033
std         0.035212
min         0.471374
25%         0.575244
50%         0.596774
75%         0.620518
max         0.782317
Name: matching_score, dtype: float64


In [13]:
print((df_notes["matching_score"] >= 0.50).sum())
print((df_notes["matching_score"] >= 0.48).sum())

4989
4995


In [14]:
selected_df_notes = df_notes[df_notes["matching_score"] >= 0.50]
print(selected_df_notes.shape)

(4989, 9)


In [16]:
selected_df_notes.to_csv("data/selected_labeled_notes.csv", index=False)